In [1]:
import os
import cv2
from tqdm import tqdm

RAW_DIR = "YOLO_Dataset/images/val"

HE_DIR = "Results/HE_images"
CLAHE_DIR = "Results/CLAHE_images"

os.makedirs(HE_DIR, exist_ok=True)
os.makedirs(CLAHE_DIR, exist_ok=True)

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8,8)
)

images = [
    f for f in os.listdir(RAW_DIR)
    if f.lower().endswith((".jpg",".jpeg",".png"))
]

for img_name in tqdm(images):

    img_path = os.path.join(RAW_DIR, img_name)

    img = cv2.imread(img_path)

    if img is None:
        continue

    # ------------------------
    # HE
    # ------------------------

    ycrcb = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2YCrCb
    )

    ycrcb[:,:,0] = cv2.equalizeHist(
        ycrcb[:,:,0]
    )

    he_img = cv2.cvtColor(
        ycrcb,
        cv2.COLOR_YCrCb2BGR
    )

    cv2.imwrite(
        os.path.join(
            HE_DIR,
            img_name
        ),
        he_img
    )

    # ------------------------
    # CLAHE
    # ------------------------

    ycrcb = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2YCrCb
    )

    ycrcb[:,:,0] = clahe.apply(
        ycrcb[:,:,0]
    )

    clahe_img = cv2.cvtColor(
        ycrcb,
        cv2.COLOR_YCrCb2BGR
    )

    cv2.imwrite(
        os.path.join(
            CLAHE_DIR,
            img_name
        ),
        clahe_img
    )

print("Done")

100%|█████████████████████████████████████████████████████████████████████████████| 1147/1147 [00:08<00:00, 138.14it/s]

Done


In [2]:
import os
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

# ------------------------------------------------
# PATHS
# ------------------------------------------------

RAW_DIR = "YOLO_Dataset/images/val"

METHODS = {
    "HE":"Results/HE_images",
    "CLAHE":"Results/CLAHE_images",
    "FUNIEGAN":"Results/enhanced_images"
}

# ------------------------------------------------
# METRICS
# ------------------------------------------------

def brightness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.mean(gray)

def contrast(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.std(gray)

def sharpness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

# ------------------------------------------------

all_results = []

files = os.listdir(RAW_DIR)

for method, method_dir in METHODS.items():

    print(method)

    for file in tqdm(files):

        raw_path = os.path.join(
            RAW_DIR,
            file
        )

        enhanced_path = os.path.join(
            method_dir,
            file
        )

        if not os.path.exists(
            enhanced_path
        ):
            continue

        raw = cv2.imread(raw_path)
        enh = cv2.imread(enhanced_path)

        if raw is None or enh is None:
            continue

        raw_rgb = cv2.cvtColor(
            raw,
            cv2.COLOR_BGR2RGB
        )

        enh_rgb = cv2.cvtColor(
            enh,
            cv2.COLOR_BGR2RGB
        )

        psnr = peak_signal_noise_ratio(
            raw_rgb,
            enh_rgb
        )

        ssim = structural_similarity(
            raw_rgb,
            enh_rgb,
            channel_axis=2
        )

        all_results.append([
            method,
            file,
            psnr,
            ssim,
            brightness(enh),
            contrast(enh),
            sharpness(enh)
        ])

df = pd.DataFrame(
    all_results,
    columns=[
        "Method",
        "Image",
        "PSNR",
        "SSIM",
        "Brightness",
        "Contrast",
        "Sharpness"
    ]
)

df.to_csv(
    "Results/comparison_metrics.csv",
    index=False
)

df.head()

HE


100%|██████████████████████████████████████████████████████████████████████████████| 1147/1147 [01:06<00:00, 17.13it/s]


CLAHE


100%|██████████████████████████████████████████████████████████████████████████████| 1147/1147 [00:58<00:00, 19.49it/s]


FUNIEGAN


  0%|                                                                                         | 0/1147 [00:00<?, ?it/s]


ValueError: Input images must have the same dimensions.

In [3]:
print(raw.shape)
print(enh.shape)

(270, 480, 3)
(256, 256, 3)


In [5]:
raw_rgb = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)

enh_rgb = cv2.cvtColor(enh, cv2.COLOR_BGR2RGB)

In [7]:
# Make dimensions identical

if raw.shape[:2] != enh.shape[:2]:

    enh = cv2.resize(
        enh,
        (raw.shape[1], raw.shape[0]),
        interpolation=cv2.INTER_CUBIC
    )

raw_rgb = cv2.cvtColor(
    raw,
    cv2.COLOR_BGR2RGB
)

enh_rgb = cv2.cvtColor(
    enh,
    cv2.COLOR_BGR2RGB
)

In [8]:
if raw_rgb.shape != enh_rgb.shape:

    print("Mismatch:", file)
    print(raw_rgb.shape)
    print(enh_rgb.shape)

    continue

SyntaxError: 'continue' not properly in loop (1492625731.py, line 7)

In [13]:
import os
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

# ------------------------------------------------
# PATHS
# ------------------------------------------------

RAW_DIR = "YOLO_Dataset/images/val"

METHODS = {
    "HE":"Results/HE_images",
    "CLAHE":"Results/CLAHE_images",
    "FUNIEGAN":"Results/enhanced_images"
}

# ------------------------------------------------
# METRICS
# ------------------------------------------------

def brightness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.mean(gray)

def contrast(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.std(gray)

def sharpness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

# ------------------------------------------------

all_results = []

files = os.listdir(RAW_DIR)

for method, method_dir in METHODS.items():

    print(method)

    for file in tqdm(files):

        raw_path = os.path.join(
            RAW_DIR,
            file
        )

        enhanced_path = os.path.join(
            method_dir,
            file
        )

        if not os.path.exists(
            enhanced_path
        ):
            continue

        raw = cv2.imread(raw_path)
        enh = cv2.imread(enhanced_path)

        # Resize enhanced image if dimensions differ

            if raw.shape[:2] != enh.shape[:2]:
            
                enh = cv2.resize(
                    enh,
                    (raw.shape[1], raw.shape[0]),
                    interpolation=cv2.INTER_CUBIC
                )
            
            raw_rgb = cv2.cvtColor(
                raw,
                cv2.COLOR_BGR2RGB
            )
            
            enh_rgb = cv2.cvtColor(
                enh,
                cv2.COLOR_BGR2RGB
            )
        psnr = peak_signal_noise_ratio(
            raw_rgb,
            enh_rgb
        )

        ssim = structural_similarity(
            raw_rgb,
            enh_rgb,
            channel_axis=2
        )

        all_results.append([
            method,
            file,
            psnr,
            ssim,
            brightness(enh),
            contrast(enh),
            sharpness(enh)
        ])

df = pd.DataFrame(
    all_results,
    columns=[
        "Method",
        "Image",
        "PSNR",
        "SSIM",
        "Brightness",
        "Contrast",
        "Sharpness"
    ]
)

df.to_csv(
    "Results/comparison_metrics.csv",
    index=False
)

df.head()

IndentationError: unexpected indent (4291211626.py, line 90)

In [4]:
import os
import cv2

RAW_DIR = "YOLO_Dataset/images/val"
FUNIE_DIR = "Results/enhanced_images"

files = os.listdir(RAW_DIR)

for file in files:

    raw_path = os.path.join(RAW_DIR, file)
    enh_path = os.path.join(FUNIE_DIR, file)

    if not os.path.exists(enh_path):
        continue

    raw = cv2.imread(raw_path)
    enh = cv2.imread(enh_path)

    if raw is None or enh is None:
        continue

    if raw.shape != enh.shape:

        print("Mismatch Found")
        print(file)
        print("RAW:", raw.shape)
        print("FUNIE:", enh.shape)

        break

Mismatch Found
vid_000002_frame0000013.jpg
RAW: (270, 480, 3)
FUNIE: (256, 256, 3)


In [5]:
import os
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm

from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

In [6]:
# =====================================
# UIQM
# =====================================

def uicm(img):

    img = img.astype(np.float32)

    rg = img[:,:,2] - img[:,:,1]
    yb = 0.5*(img[:,:,2] + img[:,:,1]) - img[:,:,0]

    urg = np.mean(rg)
    uyb = np.mean(yb)

    srg = np.std(rg)
    syb = np.std(yb)

    return (
        -0.0268*np.sqrt(urg**2 + uyb**2)
        +
        0.1586*np.sqrt(srg**2 + syb**2)
    )

def uism(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    sobelx = cv2.Sobel(
        gray,
        cv2.CV_64F,
        1,
        0,
        ksize=3
    )

    sobely = cv2.Sobel(
        gray,
        cv2.CV_64F,
        0,
        1,
        ksize=3
    )

    edge = np.sqrt(
        sobelx**2 +
        sobely**2
    )

    return np.mean(edge)

def uiconm(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.std(gray)

def uiqm(img):

    return (
        0.0282 * uicm(img)
        +
        0.2953 * uism(img)
        +
        3.5753 * uiconm(img)
    )

In [16]:
def brightness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.mean(gray)

def contrast(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.std(gray)

def sharpness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

In [7]:
RAW_DIR = "YOLO_Dataset/images/val"

METHODS = {

    "HE":
    "Results/HE_images",

    "CLAHE":
    "Results/CLAHE_images",

    "FUNIEGAN":
    "Results/enhanced_images"
}

In [8]:
all_results = []

files = os.listdir(RAW_DIR)

for method, method_dir in METHODS.items():

    print("\nProcessing:", method)

    for file in tqdm(files):

        raw_path = os.path.join(
            RAW_DIR,
            file
        )

        enhanced_path = os.path.join(
            method_dir,
            file
        )

        if not os.path.exists(
            enhanced_path
        ):
            continue

        raw = cv2.imread(raw_path)
        enh = cv2.imread(enhanced_path)

        if raw is None:
            continue

        if enh is None:
            continue

        # --------------------------
        # Fix dimension mismatch
        # --------------------------

        if raw.shape[:2] != enh.shape[:2]:

            enh = cv2.resize(
                enh,
                (
                    raw.shape[1],
                    raw.shape[0]
                ),
                interpolation=cv2.INTER_CUBIC
            )

        raw_rgb = cv2.cvtColor(
            raw,
            cv2.COLOR_BGR2RGB
        )

        enh_rgb = cv2.cvtColor(
            enh,
            cv2.COLOR_BGR2RGB
        )

        # --------------------------
        # Safety check
        # --------------------------

        if raw_rgb.shape != enh_rgb.shape:

            print(
                "Shape mismatch:",
                file
            )

            continue

        psnr = peak_signal_noise_ratio(
            raw_rgb,
            enh_rgb,
            data_range=255
        )

        ssim = structural_similarity(
            raw_rgb,
            enh_rgb,
            channel_axis=2,
            data_range=255
        )

        all_results.append([

            method,
            file,

            psnr,
            ssim,

            uiqm(enh),

            brightness(enh),
            contrast(enh),
            sharpness(enh)

        ])

df = pd.DataFrame(

    all_results,

    columns=[

        "Method",
        "Image",

        "PSNR",
        "SSIM",
        "UIQM",

        "Brightness",
        "Contrast",
        "Sharpness"
    ]
)

df.to_csv(

    "Results/comparison_metrics.csv",

    index=False
)

print(df.head())


Processing: HE


  0%|                                                                                         | 0/1147 [00:00<?, ?it/s]


NameError: name 'brightness' is not defined

In [9]:
import os
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm

from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

In [10]:
RAW_DIR = "Dataset/raw"

HE_DIR = "Results/HE_images"
CLAHE_DIR = "Results/CLAHE_images"
FUNIE_DIR = "Results/enhanced_images"

print("Raw Images:", len(os.listdir(RAW_DIR)))

print("FUNIE Images:", len(os.listdir(FUNIE_DIR)))

Raw Images: 0
FUNIE Images: 7212


In [11]:
import os

print(os.listdir("."))

['.ipynb_checkpoints', '01_MDEEF_EnhancementPipeline.ipynb', 'Dataset', 'Figure1_MDEEF_Framework.png', 'Figure2_Benchmark_Workflow.png', 'Figure3_Detection_Module.png', 'Figure4_MDEEF_Contribution.png', 'Figure5A_Strong_Enhancement.png', 'Figures', 'Figures.ipynb', 'Models', 'Notebooks', 'Other Models.ipynb', 'Results', 'runs', 'TrashCan YOLOv11 Converter.ipynb', 'yolo11n.pt', 'YOLOv11 Training.ipynb', 'YOLO_Dataset']


In [12]:
import os

print(os.listdir("Dataset"))

['enhanced', 'metrics', 'raw']


In [13]:
import os

for item in os.listdir("Dataset"):

    path = os.path.join("Dataset", item)

    if os.path.isdir(path):

        print(item, len(os.listdir(path)))

enhanced 0
metrics 0
raw 0


In [14]:
FUNIE_DIR = "Results/enhanced_images"

files = os.listdir(FUNIE_DIR)

print(len(files))

7212


In [15]:
TRAIN_DIR = "YOLO_Dataset/images/train"
VAL_DIR = "YOLO_Dataset/images/val"

In [16]:
def find_raw_image(filename):

    train_path = os.path.join(
        TRAIN_DIR,
        filename
    )

    val_path = os.path.join(
        VAL_DIR,
        filename
    )

    if os.path.exists(train_path):
        return train_path

    if os.path.exists(val_path):
        return val_path

    return None

In [17]:
import os

print(
    "Train:",
    len(os.listdir("YOLO_Dataset/images/train"))
)

print(
    "Val:",
    len(os.listdir("YOLO_Dataset/images/val"))
)

Train: 6065
Val: 1147


In [18]:
import os
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm

from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

In [19]:
TRAIN_DIR = "YOLO_Dataset/images/train"
VAL_DIR   = "YOLO_Dataset/images/val"

HE_DIR     = "Results/HE_images"
CLAHE_DIR  = "Results/CLAHE_images"
FUNIE_DIR  = "Results/enhanced_images"

print("Train:", len(os.listdir(TRAIN_DIR)))
print("Val:", len(os.listdir(VAL_DIR)))

print(
    "Total Images:",
    len(os.listdir(TRAIN_DIR))
    +
    len(os.listdir(VAL_DIR))
)

Train: 6065
Val: 1147
Total Images: 7212


In [20]:
def find_raw_image(filename):

    train_path = os.path.join(
        TRAIN_DIR,
        filename
    )

    val_path = os.path.join(
        VAL_DIR,
        filename
    )

    if os.path.exists(train_path):
        return train_path

    if os.path.exists(val_path):
        return val_path

    return None

In [21]:
def uicm(img):

    img = img.astype(np.float32)

    rg = img[:,:,2] - img[:,:,1]

    yb = (
        0.5*(img[:,:,2] + img[:,:,1])
        - img[:,:,0]
    )

    urg = np.mean(rg)
    uyb = np.mean(yb)

    srg = np.std(rg)
    syb = np.std(yb)

    return (
        -0.0268*np.sqrt(
            urg**2 + uyb**2
        )
        +
        0.1586*np.sqrt(
            srg**2 + syb**2
        )
    )

def uism(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    sobelx = cv2.Sobel(
        gray,
        cv2.CV_64F,
        1,
        0,
        ksize=3
    )

    sobely = cv2.Sobel(
        gray,
        cv2.CV_64F,
        0,
        1,
        ksize=3
    )

    edge = np.sqrt(
        sobelx**2 +
        sobely**2
    )

    return np.mean(edge)

def uiconm(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.std(gray)

def uiqm(img):

    return (
        0.0282*uicm(img)
        +
        0.2953*uism(img)
        +
        3.5753*uiconm(img)
    )

In [22]:
def brightness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.mean(gray)

def contrast(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.std(gray)

def sharpness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

In [24]:
METHODS = {

    "HE": HE_DIR,

    "CLAHE": CLAHE_DIR,

    "FUNIEGAN": FUNIE_DIR
}

all_results = []

for method, method_dir in METHODS.items():

    print("\nProcessing:", method)

    files = os.listdir(method_dir)

    for file in tqdm(files):

        raw_path = find_raw_image(file)

        if raw_path is None:
            continue

        enhanced_path = os.path.join(
            method_dir,
            file
        )

        raw = cv2.imread(raw_path)
        enh = cv2.imread(enhanced_path)

        if raw is None or enh is None:
            continue

        if raw.shape[:2] != enh.shape[:2]:

            enh = cv2.resize(
                enh,
                (
                    raw.shape[1],
                    raw.shape[0]
                ),
                interpolation=cv2.INTER_CUBIC
            )

        raw_rgb = cv2.cvtColor(
            raw,
            cv2.COLOR_BGR2RGB
        )

        enh_rgb = cv2.cvtColor(
            enh,
            cv2.COLOR_BGR2RGB
        )

        psnr = peak_signal_noise_ratio(
            raw_rgb,
            enh_rgb,
            data_range=255
        )

        ssim = structural_similarity(
            raw_rgb,
            enh_rgb,
            channel_axis=2,
            data_range=255
        )

        all_results.append([

            method,
            file,

            psnr,
            ssim,

            uiqm(enh),

            brightness(enh),

            contrast(enh),

            sharpness(enh)
        ])

df = pd.DataFrame(

    all_results,

    columns=[

        "Method",
        "Image",

        "PSNR",
        "SSIM",

        "UIQM",

        "Brightness",

        "Contrast",

        "Sharpness"
    ]
)

df.to_csv(
    "Results/comparison_metrics_7212.csv",
    index=False
)

print(df.head())


Processing: HE


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'Results/HE_images'

In [25]:
import os
import cv2
from tqdm import tqdm

TRAIN_DIR = "YOLO_Dataset/images/train"
VAL_DIR   = "YOLO_Dataset/images/val"

HE_DIR = "Results/HE_images"
CLAHE_DIR = "Results/CLAHE_images"

os.makedirs(HE_DIR, exist_ok=True)
os.makedirs(CLAHE_DIR, exist_ok=True)

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8,8)
)

all_images = []

for folder in [TRAIN_DIR, VAL_DIR]:

    for img in os.listdir(folder):

        all_images.append(
            os.path.join(folder, img)
        )

print("Total:", len(all_images))

Total: 7212


In [26]:
print("HE:", len(os.listdir("Results/HE_images")))
print("CLAHE:", len(os.listdir("Results/CLAHE_images")))
print("FUNIE:", len(os.listdir("Results/enhanced_images")))

HE: 0
CLAHE: 0
FUNIE: 7212


In [27]:
import os
import cv2
from tqdm import tqdm

TRAIN_DIR = "YOLO_Dataset/images/train"
VAL_DIR   = "YOLO_Dataset/images/val"

HE_DIR = "Results/HE_images"
CLAHE_DIR = "Results/CLAHE_images"

os.makedirs(HE_DIR, exist_ok=True)
os.makedirs(CLAHE_DIR, exist_ok=True)

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8,8)
)

all_images = []

for folder in [TRAIN_DIR, VAL_DIR]:

    for img in os.listdir(folder):

        if img.lower().endswith(
            (".jpg",".jpeg",".png")
        ):
            all_images.append(
                os.path.join(folder, img)
            )

print("Total Images:", len(all_images))

for img_path in tqdm(all_images):

    img_name = os.path.basename(img_path)

    img = cv2.imread(img_path)

    if img is None:
        continue

    # =====================
    # HE
    # =====================

    ycrcb = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2YCrCb
    )

    ycrcb[:,:,0] = cv2.equalizeHist(
        ycrcb[:,:,0]
    )

    he_img = cv2.cvtColor(
        ycrcb,
        cv2.COLOR_YCrCb2BGR
    )

    cv2.imwrite(
        os.path.join(
            HE_DIR,
            img_name
        ),
        he_img
    )

    # =====================
    # CLAHE
    # =====================

    ycrcb = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2YCrCb
    )

    ycrcb[:,:,0] = clahe.apply(
        ycrcb[:,:,0]
    )

    clahe_img = cv2.cvtColor(
        ycrcb,
        cv2.COLOR_YCrCb2BGR
    )

    cv2.imwrite(
        os.path.join(
            CLAHE_DIR,
            img_name
        ),
        clahe_img
    )

print("DONE")

Total Images: 7212


100%|██████████████████████████████████████████████████████████████████████████████| 7212/7212 [01:43<00:00, 69.98it/s]

DONE


In [28]:
import os

print("HE:", len(os.listdir("Results/HE_images")))
print("CLAHE:", len(os.listdir("Results/CLAHE_images")))
print("FUNIE:", len(os.listdir("Results/enhanced_images")))

HE: 7212
CLAHE: 7212
FUNIE: 7212


In [29]:
import os
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm

from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)

In [30]:
TRAIN_DIR = "YOLO_Dataset/images/train"
VAL_DIR   = "YOLO_Dataset/images/val"

HE_DIR = "Results/HE_images"
CLAHE_DIR = "Results/CLAHE_images"
FUNIE_DIR = "Results/enhanced_images"

print("Train:", len(os.listdir(TRAIN_DIR)))
print("Val:", len(os.listdir(VAL_DIR)))

print("HE:", len(os.listdir(HE_DIR)))
print("CLAHE:", len(os.listdir(CLAHE_DIR)))
print("FUNIE:", len(os.listdir(FUNIE_DIR)))

Train: 6065
Val: 1147
HE: 7212
CLAHE: 7212
FUNIE: 7212


In [31]:
def find_raw_image(filename):

    train_path = os.path.join(
        TRAIN_DIR,
        filename
    )

    val_path = os.path.join(
        VAL_DIR,
        filename
    )

    if os.path.exists(train_path):
        return train_path

    if os.path.exists(val_path):
        return val_path

    return None

In [32]:
def uicm(img):

    img = img.astype(np.float32)

    rg = img[:,:,2] - img[:,:,1]

    yb = (
        0.5*(img[:,:,2] + img[:,:,1])
        - img[:,:,0]
    )

    urg = np.mean(rg)
    uyb = np.mean(yb)

    srg = np.std(rg)
    syb = np.std(yb)

    return (
        -0.0268*np.sqrt(
            urg**2 + uyb**2
        )
        +
        0.1586*np.sqrt(
            srg**2 + syb**2
        )
    )

def uism(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    sobelx = cv2.Sobel(
        gray,
        cv2.CV_64F,
        1,
        0,
        ksize=3
    )

    sobely = cv2.Sobel(
        gray,
        cv2.CV_64F,
        0,
        1,
        ksize=3
    )

    edge = np.sqrt(
        sobelx**2 +
        sobely**2
    )

    return np.mean(edge)

def uiconm(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.std(gray)

def uiqm(img):

    return (
        0.0282*uicm(img)
        +
        0.2953*uism(img)
        +
        3.5753*uiconm(img)
    )

In [33]:
def brightness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.mean(gray)

def contrast(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.std(gray)

def sharpness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

In [34]:
METHODS = {

    "HE": HE_DIR,

    "CLAHE": CLAHE_DIR,

    "FUNIEGAN": FUNIE_DIR
}

all_results = []

for method, method_dir in METHODS.items():

    print(f"\nProcessing {method}")

    files = os.listdir(method_dir)

    for file in tqdm(files):

        raw_path = find_raw_image(file)

        if raw_path is None:
            continue

        enhanced_path = os.path.join(
            method_dir,
            file
        )

        raw = cv2.imread(raw_path)
        enh = cv2.imread(enhanced_path)

        if raw is None or enh is None:
            continue

        if raw.shape[:2] != enh.shape[:2]:

            enh = cv2.resize(
                enh,
                (
                    raw.shape[1],
                    raw.shape[0]
                ),
                interpolation=cv2.INTER_CUBIC
            )

        raw_rgb = cv2.cvtColor(
            raw,
            cv2.COLOR_BGR2RGB
        )

        enh_rgb = cv2.cvtColor(
            enh,
            cv2.COLOR_BGR2RGB
        )

        psnr = peak_signal_noise_ratio(
            raw_rgb,
            enh_rgb,
            data_range=255
        )

        ssim = structural_similarity(
            raw_rgb,
            enh_rgb,
            channel_axis=2,
            data_range=255
        )

        all_results.append([

            method,
            file,

            psnr,
            ssim,

            uiqm(enh),

            brightness(enh),

            contrast(enh),

            sharpness(enh)

        ])

df = pd.DataFrame(

    all_results,

    columns=[

        "Method",
        "Image",

        "PSNR",
        "SSIM",

        "UIQM",

        "Brightness",

        "Contrast",

        "Sharpness"
    ]
)

df.to_csv(
    "Results/comparison_metrics_7212.csv",
    index=False
)

print(df.head())


Processing HE


100%|██████████████████████████████████████████████████████████████████████████████| 7212/7212 [07:04<00:00, 16.99it/s]



Processing CLAHE


100%|██████████████████████████████████████████████████████████████████████████████| 7212/7212 [06:42<00:00, 17.91it/s]



Processing FUNIEGAN


100%|██████████████████████████████████████████████████████████████████████████████| 7212/7212 [06:24<00:00, 18.75it/s]


  Method                        Image       PSNR      SSIM        UIQM  \
0     HE  vid_000002_frame0000013.jpg  11.052217  0.541499  280.311949   
1     HE  vid_000002_frame0000014.jpg  10.735148  0.496956  282.010757   
2     HE  vid_000002_frame0000015.jpg  10.347363  0.454648  284.540536   
3     HE  vid_000002_frame0000016.jpg  10.020870  0.437503  284.881117   
4     HE  vid_000002_frame0000017.jpg   9.897302  0.400580  286.174906   

   Brightness   Contrast     Sharpness  
0  130.330355  72.730823   7068.053059  
1  130.593711  72.659427   7321.076235  
2  131.165664  72.618836   9029.113687  
3  132.001019  72.432857  10659.456538  
4  132.331806  72.404436  10603.345749  


In [35]:
summary = (

    df.groupby("Method")

      .mean(numeric_only=True)

      .round(3)

)

summary

,PSNR,SSIM,UIQM,Brightness,Contrast,Sharpness
Method,,,,,,
CLAHE,24.925,0.861,155.249,100.424,39.115,2574.850
FUNIEGAN,17.606,0.702,181.559,113.181,44.309,3293.833
HE,13.203,0.664,287.643,124.012,74.998,4599.711


In [36]:
summary.to_csv(
    "Results/comparison_summary_7212.csv"
)

summary

,PSNR,SSIM,UIQM,Brightness,Contrast,Sharpness
Method,,,,,,
CLAHE,24.925,0.861,155.249,100.424,39.115,2574.850
FUNIEGAN,17.606,0.702,181.559,113.181,44.309,3293.833
HE,13.203,0.664,287.643,124.012,74.998,4599.711


In [37]:
import numpy as np
import pandas as pd

def cohens_d_paired(before, after):

    diff = after - before

    return np.mean(diff) / np.std(diff, ddof=1)

metrics = [
    ("Brightness_Raw", "Brightness_Enhanced"),
    ("Contrast_Raw", "Contrast_Enhanced"),
    ("Sharpness_Raw", "Sharpness_Enhanced")
]

for raw_col, enh_col in metrics:

    d = cohens_d_paired(
        df[raw_col],
        df[enh_col]
    )

    print(
        f"{raw_col.split('_')[0]}: "
        f"Cohen's d = {d:.4f}"
    )

KeyError: 'Brightness_Raw'

In [38]:
print(df.columns.tolist())

['Method', 'Image', 'PSNR', 'SSIM', 'UIQM', 'Brightness', 'Contrast', 'Sharpness']


In [39]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

TRAIN_DIR = "YOLO_Dataset/images/train"
VAL_DIR   = "YOLO_Dataset/images/val"

FUNIE_DIR = "Results/enhanced_images"

def brightness(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return np.mean(gray)

def contrast(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return np.std(gray)

def sharpness(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def find_raw_image(filename):

    train_path = os.path.join(TRAIN_DIR, filename)

    if os.path.exists(train_path):
        return train_path

    val_path = os.path.join(VAL_DIR, filename)

    if os.path.exists(val_path):
        return val_path

    return None

results = []

files = os.listdir(FUNIE_DIR)

for file in tqdm(files):

    raw_path = find_raw_image(file)

    if raw_path is None:
        continue

    enh_path = os.path.join(FUNIE_DIR, file)

    raw = cv2.imread(raw_path)
    enh = cv2.imread(enh_path)

    if raw is None or enh is None:
        continue

    if raw.shape[:2] != enh.shape[:2]:

        enh = cv2.resize(
            enh,
            (raw.shape[1], raw.shape[0])
        )

    results.append([

        file,

        brightness(raw),
        brightness(enh),

        contrast(raw),
        contrast(enh),

        sharpness(raw),
        sharpness(enh)

    ])

stats_df = pd.DataFrame(

    results,

    columns=[

        "Image",

        "Brightness_Raw",
        "Brightness_Enhanced",

        "Contrast_Raw",
        "Contrast_Enhanced",

        "Sharpness_Raw",
        "Sharpness_Enhanced"
    ]
)

stats_df.to_csv(
    "Results/statistical_metrics.csv",
    index=False
)

stats_df.head()

100%|██████████████████████████████████████████████████████████████████████████████| 7212/7212 [01:20<00:00, 89.13it/s]


,Image,Brightness_Raw,Brightness_Enhanced,Contrast_Raw,Contrast_Enhanced,Sharpness_Raw,Sharpness_Enhanced
0,vid_000002_frame0000013.jpg,90.361435,112.799591,19.666351,23.051287,2554.711232,2335.673074
1,vid_000002_frame0000014.jpg,87.260656,108.821427,19.158107,22.696007,2514.444799,2252.358377
2,vid_000002_frame0000015.jpg,84.000023,107.856520,18.706731,22.712005,2645.198693,2431.895585
3,vid_000002_frame0000016.jpg,81.288480,105.786991,18.281672,22.403915,2728.245563,2467.117481
4,vid_000002_frame0000017.jpg,80.351752,105.010363,18.255266,21.980982,2652.106083,2413.531168


In [40]:
import numpy as np
import pandas as pd

df = pd.read_csv(
    "Results/statistical_metrics.csv"
)

def cohens_d_paired(before, after):

    diff = after - before

    return np.mean(diff) / np.std(diff, ddof=1)

for metric in [

    "Brightness",
    "Contrast",
    "Sharpness"

]:

    d = cohens_d_paired(

        df[f"{metric}_Raw"],

        df[f"{metric}_Enhanced"]

    )

    print(
        f"{metric}: Cohen's d = {d:.4f}"
    )

Brightness: Cohen's d = 2.6170
Contrast: Cohen's d = 0.6590
Sharpness: Cohen's d = 0.1465


In [41]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

TRAIN_DIR = "YOLO_Dataset/images/train"
VAL_DIR   = "YOLO_Dataset/images/val"

FUNIE_DIR = "Results/enhanced_images"

# -----------------------------------
# Metrics
# -----------------------------------

def brightness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.mean(gray)

def contrast(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.std(gray)

# -----------------------------------

def find_raw_image(filename):

    train_path = os.path.join(
        TRAIN_DIR,
        filename
    )

    if os.path.exists(train_path):
        return train_path

    val_path = os.path.join(
        VAL_DIR,
        filename
    )

    if os.path.exists(val_path):
        return val_path

    return None

# -----------------------------------

records = []

files = os.listdir(FUNIE_DIR)

for file in tqdm(files):

    raw_path = find_raw_image(file)

    if raw_path is None:
        continue

    enh_path = os.path.join(
        FUNIE_DIR,
        file
    )

    raw = cv2.imread(raw_path)
    enh = cv2.imread(enh_path)

    if raw is None or enh is None:
        continue

    if raw.shape[:2] != enh.shape[:2]:

        enh = cv2.resize(
            enh,
            (
                raw.shape[1],
                raw.shape[0]
            )
        )

    records.append([

        file,

        brightness(raw),
        brightness(enh),

        contrast(raw),
        contrast(enh)

    ])

df = pd.DataFrame(

    records,

    columns=[

        "Image",

        "Brightness_Raw",
        "Brightness_Enhanced",

        "Contrast_Raw",
        "Contrast_Enhanced"
    ]
)

# -----------------------------------
# Turbidity Estimation
# -----------------------------------

df["Turbidity_Group"] = pd.qcut(

    df["Contrast_Raw"],

    q=3,

    labels=[

        "High Turbidity",
        "Medium Turbidity",
        "Low Turbidity"
    ]
)

# -----------------------------------
# Improvements
# -----------------------------------

df["Brightness_Improvement"] = (

    (
        df["Brightness_Enhanced"]
        -
        df["Brightness_Raw"]
    )

    /

    df["Brightness_Raw"]

) * 100

df["Contrast_Improvement"] = (

    (
        df["Contrast_Enhanced"]
        -
        df["Contrast_Raw"]
    )

    /

    df["Contrast_Raw"]

) * 100

summary = (

    df.groupby(
        "Turbidity_Group"
    )

    .agg({

        "Image":"count",

        "Brightness_Improvement":"mean",

        "Contrast_Improvement":"mean"

    })

    .round(2)

)

summary

100%|█████████████████████████████████████████████████████████████████████████████| 7212/7212 [00:52<00:00, 136.99it/s]
C:\Users\rjrah\AppData\Local\Temp\ipykernel_34900\2351976415.py:170: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(


,Image,Brightness_Improvement,Contrast_Improvement
Turbidity_Group,,,
High Turbidity,2404,26.38,24.91
Medium Turbidity,2404,28.71,18.21
Low Turbidity,2404,20.52,4.68


In [42]:
import numpy as np
from scipy.stats import bootstrap

def cohens_d_paired(before, after):

    diff = after - before

    return np.mean(diff) / np.std(diff, ddof=1)

def bootstrap_ci(before, after):

    diff = np.array(after) - np.array(before)

    res = bootstrap(
        (diff,),
        np.mean,
        confidence_level=0.95,
        n_resamples=5000,
        method='percentile'
    )

    return (
        res.confidence_interval.low,
        res.confidence_interval.high
    )

metrics = [

    "Brightness",
    "Contrast",
    "Sharpness"

]

for metric in metrics:

    d = cohens_d_paired(
        df[f"{metric}_Raw"],
        df[f"{metric}_Enhanced"]
    )

    low, high = bootstrap_ci(
        df[f"{metric}_Raw"],
        df[f"{metric}_Enhanced"]
    )

    print(
        metric,
        round(d,3),
        round(low,3),
        round(high,3)
    )

Brightness 2.617 18.72 19.053
Contrast 0.659 4.374 4.699


KeyError: 'Sharpness_Raw'

In [43]:
print(df.columns.tolist())

['Image', 'Brightness_Raw', 'Brightness_Enhanced', 'Contrast_Raw', 'Contrast_Enhanced', 'Turbidity_Group', 'Brightness_Improvement', 'Contrast_Improvement']


In [45]:
from scipy.stats import bootstrap
import numpy as np

def cohens_d_paired(before, after):

    diff = after - before

    return np.mean(diff) / np.std(diff, ddof=1)

def ci_mean_diff(before, after):

    diff = np.array(after) - np.array(before)

    result = bootstrap(
        (diff,),
        np.mean,
        confidence_level=0.95,
        n_resamples=5000,
        method="percentile"
    )

    return (
        result.confidence_interval.low,
        result.confidence_interval.high
    )

for metric in ["Brightness", "Contrast"]:

    d = cohens_d_paired(
        df[f"{metric}_Raw"],
        df[f"{metric}_Enhanced"]
    )

    low, high = ci_mean_diff(
        df[f"{metric}_Raw"],
        df[f"{metric}_Enhanced"]
    )

    print(
        f"{metric}"
    )

    print(
        f"Cohen's d = {d:.3f}"
    )

    print(
        f"95% CI = [{low:.3f}, {high:.3f}]"
    )

    print()

Brightness
Cohen's d = 2.617
95% CI = [18.715, 19.051]

Contrast
Cohen's d = 0.659
95% CI = [4.373, 4.692]



In [46]:
Sharpness_Raw
Sharpness_Enhanced

NameError: name 'Sharpness_Raw' is not defined

In [47]:
statistical_metrics.csv

NameError: name 'statistical_metrics' is not defined

In [48]:
def sharpness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

In [49]:
results.append([

    file,

    brightness(raw),
    brightness(enh),

    contrast(raw),
    contrast(enh),

    sharpness(raw),
    sharpness(enh)

])

In [50]:
columns=[

    "Image",

    "Brightness_Raw",
    "Brightness_Enhanced",

    "Contrast_Raw",
    "Contrast_Enhanced",

    "Sharpness_Raw",
    "Sharpness_Enhanced"

]

In [53]:
import pandas as pd

df = pd.read_csv(
    "Results/statistical_metrics_full.csv"
)

FileNotFoundError: [Errno 2] No such file or directory: 'Results/statistical_metrics_full.csv'

In [54]:
import os

print(
    os.path.exists(
        "Results/statistical_metrics_full.csv"
    )
)

False


In [55]:
print(df.columns.tolist())

['Image', 'Brightness_Raw', 'Brightness_Enhanced', 'Contrast_Raw', 'Contrast_Enhanced', 'Turbidity_Group', 'Brightness_Improvement', 'Contrast_Improvement']


In [56]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

TRAIN_DIR = "YOLO_Dataset/images/train"
VAL_DIR   = "YOLO_Dataset/images/val"

FUNIE_DIR = "Results/enhanced_images"

def brightness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.mean(gray)

def contrast(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return np.std(gray)

def sharpness(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    return cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

def find_raw_image(filename):

    train_path = os.path.join(
        TRAIN_DIR,
        filename
    )

    if os.path.exists(train_path):
        return train_path

    val_path = os.path.join(
        VAL_DIR,
        filename
    )

    if os.path.exists(val_path):
        return val_path

    return None

records = []

files = os.listdir(FUNIE_DIR)

for file in tqdm(files):

    raw_path = find_raw_image(file)

    if raw_path is None:
        continue

    enh_path = os.path.join(
        FUNIE_DIR,
        file
    )

    raw = cv2.imread(raw_path)
    enh = cv2.imread(enh_path)

    if raw is None or enh is None:
        continue

    if raw.shape[:2] != enh.shape[:2]:

        enh = cv2.resize(
            enh,
            (
                raw.shape[1],
                raw.shape[0]
            )
        )

    records.append([

        file,

        brightness(raw),
        brightness(enh),

        contrast(raw),
        contrast(enh),

        sharpness(raw),
        sharpness(enh)

    ])

df_stats = pd.DataFrame(

    records,

    columns=[

        "Image",

        "Brightness_Raw",
        "Brightness_Enhanced",

        "Contrast_Raw",
        "Contrast_Enhanced",

        "Sharpness_Raw",
        "Sharpness_Enhanced"
    ]
)

df_stats.to_csv(

    "Results/statistical_metrics_full.csv",

    index=False
)

print(df_stats.columns.tolist())
print(df_stats.shape)

100%|██████████████████████████████████████████████████████████████████████████████| 7212/7212 [01:20<00:00, 89.34it/s]


['Image', 'Brightness_Raw', 'Brightness_Enhanced', 'Contrast_Raw', 'Contrast_Enhanced', 'Sharpness_Raw', 'Sharpness_Enhanced']
(7212, 7)


In [57]:
import pandas as pd

df = pd.read_csv(
    "Results/statistical_metrics_full.csv"
)

print(df.columns.tolist())

['Image', 'Brightness_Raw', 'Brightness_Enhanced', 'Contrast_Raw', 'Contrast_Enhanced', 'Sharpness_Raw', 'Sharpness_Enhanced']


In [58]:
from scipy.stats import bootstrap
import numpy as np

def cohens_d_paired(before, after):

    diff = after - before

    return np.mean(diff) / np.std(diff, ddof=1)

def ci_mean_diff(before, after):

    diff = np.array(after) - np.array(before)

    result = bootstrap(
        (diff,),
        np.mean,
        confidence_level=0.95,
        n_resamples=5000,
        method='percentile'
    )

    return (
        result.confidence_interval.low,
        result.confidence_interval.high
    )

for metric in [

    "Brightness",
    "Contrast",
    "Sharpness"

]:

    d = cohens_d_paired(
        df[f"{metric}_Raw"],
        df[f"{metric}_Enhanced"]
    )

    low, high = ci_mean_diff(
        df[f"{metric}_Raw"],
        df[f"{metric}_Enhanced"]
    )

    print(
        metric,
        "d =",
        round(d,3),
        "CI =",
        [round(low,3), round(high,3)]
    )

Brightness d = 2.617 CI = [np.float64(18.718), np.float64(19.05)]
Contrast d = 0.659 CI = [np.float64(4.375), np.float64(4.693)]
Sharpness d = 0.147 CI = [np.float64(127.108), np.float64(174.41)]
